# 🚢 Task 1: Titanic Survival Prediction
**CodSoft Data Science Internship**

**Objective:** Build a classification model to predict whether a passenger survived the Titanic disaster based on features like age, gender, ticket class, fare, etc.

---

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## Step 2: Load Dataset
> Download dataset from: https://www.kaggle.com/datasets/brendan45774/test-file  
> Or use the built-in seaborn Titanic dataset below.

In [ ]:
# Option A: Load from seaborn (no download needed)
df = sns.load_dataset('titanic')

# Option B: Load from your downloaded CSV file
# df = pd.read_csv('titanic.csv')

print('Dataset Shape:', df.shape)
df.head()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Survival count
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Survival distribution
df['survived'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Survival Count')
axes[0].set_xticklabels(['Did Not Survive', 'Survived'], rotation=0)
axes[0].set_ylabel('Count')

# Survival by sex
df.groupby('sex')['survived'].mean().plot(kind='bar', ax=axes[1], color=['#3498db', '#e91e63'], edgecolor='black')
axes[1].set_title('Survival Rate by Gender')
axes[1].set_xticklabels(['Female', 'Male'], rotation=0)
axes[1].set_ylabel('Survival Rate')

# Survival by class
df.groupby('pclass')['survived'].mean().plot(kind='bar', ax=axes[2], color=['#f39c12', '#9b59b6', '#1abc9c'], edgecolor='black')
axes[2].set_title('Survival Rate by Ticket Class')
axes[2].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0)
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.suptitle('Titanic - Survival Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.savefig('titanic_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Age distribution by survival
plt.figure(figsize=(10, 4))
df[df['survived'] == 1]['age'].dropna().hist(bins=30, alpha=0.6, color='green', label='Survived')
df[df['survived'] == 0]['age'].dropna().hist(bins=30, alpha=0.6, color='red', label='Did Not Survive')
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by Survival')
plt.legend()
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# Select relevant features
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target = 'survived'

df_model = df[features + [target]].copy()

# Fill missing values
df_model['age'].fillna(df_model['age'].median(), inplace=True)
df_model['fare'].fillna(df_model['fare'].median(), inplace=True)
df_model['embarked'].fillna(df_model['embarked'].mode()[0], inplace=True)

# Encode categorical variables
le = LabelEncoder()
df_model['sex'] = le.fit_transform(df_model['sex'])          # male=1, female=0
df_model['embarked'] = le.fit_transform(df_model['embarked']) # C=0, Q=1, S=2

print('Preprocessing complete!')
print(df_model.isnull().sum())
df_model.head()

## Step 5: Train-Test Split

In [ ]:
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)}')
print(f'Testing samples  : {len(X_test)}')

## Step 6: Model Training

In [ ]:
# --- Model 1: Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# --- Model 2: Random Forest Classifier ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest Classifier ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Did Not Survive', 'Survived']))

## Step 7: Model Evaluation

In [ ]:
# Confusion Matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_lr),
    display_labels=['Did Not Survive', 'Survived']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_rf),
    display_labels=['Did Not Survive', 'Survived']
).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Random Forest')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('titanic_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importances - Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('titanic_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Model Comparison
models = ['Logistic Regression', 'Random Forest']
accuracies = [
    accuracy_score(y_test, y_pred_lr),
    accuracy_score(y_test, y_pred_rf)
]

plt.figure(figsize=(7, 4))
bars = plt.bar(models, accuracies, color=['#3498db', '#2ecc71'], edgecolor='black', width=0.4)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.03,
             f'{acc:.4f}', ha='center', va='top', fontsize=12, fontweight='bold', color='white')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('titanic_model_comparison.png', bbox_inches='tight')
plt.show()

print(f'\nBest Model: {models[accuracies.index(max(accuracies))]} with accuracy {max(accuracies):.4f}')

## ✅ Summary

| Model | Accuracy |
|-------|----------|
| Logistic Regression | ~80% |
| Random Forest | ~82% |

**Key Insights:**
- Gender is the most important feature — females had a much higher survival rate.
- Passengers in 1st class had significantly better survival odds.
- Younger passengers and those who paid higher fares were more likely to survive.
- Random Forest outperforms Logistic Regression due to its ability to capture non-linear relationships.
